In [3]:
from pathlib import Path
import json
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import firwin, iirnotch, freqz, lfilter, welch, group_delay, tf2zpk
# Anggota Kelompok :
# 1. Christama Ezra Yudianto    (245150307111009)
# 2. Abdillah Fatah Suryanto    (245150300111008)
# 3. Ivang Arfandia             (245150307111003)
# 4. Jahzeel Erason Nicodemus   (245150300111002)

ROOT = Path(r"C:\Users\Erick W\OneDrive\Documents\Python\Mini Project 05")
STATUS_PATH = ROOT / "project_status.json"
CHANNEL_CFG = ROOT / "config/channel_config.json"
MP04_METRICS = ROOT / "results/metrics/mp04_channel_metrics.json"

## 1. Validasi handoff Mini Project 4

In [4]:
with open(STATUS_PATH, "r", encoding="utf-8") as f:
    status = json.load(f)
if status.get("status") not in ("ready_for_mp05", "ready_for_mp06"):
    raise RuntimeError("Mini Project 4 belum berstatus ready_for_mp05")
channel_audio = ROOT / status.get(
    "channel_audio",
    "data/processed/mp04_channel/nyala_001_channel.wav"
)
if not channel_audio.exists():
    raise FileNotFoundError("Audio channel Mini Project 4 tidak ditemukan")
if not CHANNEL_CFG.exists() or not MP04_METRICS.exists():
    raise FileNotFoundError("Konfigurasi atau metrik Mini Project 4 tidak lengkap")

## 2. Baca audio channel sebagai float -1..1

In [5]:
fs, raw = wavfile.read(channel_audio)
if raw.ndim > 1:
    raw = raw.mean(axis=1)
if np.issubdtype(raw.dtype, np.integer):
    scale = float(max(abs(np.iinfo(raw.dtype).min), np.iinfo(raw.dtype).max))
    x_channel = raw.astype(np.float32) / scale
else:
    x_channel = raw.astype(np.float32)

## 3. Tambahkan hum 50 Hz yang terkontrol

In [6]:
hum_frequency_hz = 50.0
hum_amplitude = 0.04
n = np.arange(len(x_channel))
hum = hum_amplitude * np.sin(2*np.pi*hum_frequency_hz*n/fs)
x_hum = x_channel + hum
# Sisakan headroom jika puncak terlalu besar
peak = np.max(np.abs(x_hum)) + 1e-12
if peak > 0.98:
    x_hum = x_hum * (0.98 / peak)

## 4. FIR high-pass sederhana: parameter tetap untuk MP5

In [7]:
fir_numtaps = 129
fir_cutoff_hz = 100.0
b_fir = firwin(
    fir_numtaps,
    fir_cutoff_hz,
    pass_zero=False,
    fs=fs
).astype(np.float64)
a_fir = np.array([1.0])
y_fir = lfilter(b_fir, a_fir, x_hum).astype(np.float32)
fir_group_delay_samples = (fir_numtaps - 1) // 2
fir_group_delay_ms = 1000.0 * fir_group_delay_samples / fs

## 5. IIR notch sederhana: parameter tetap untuk MP5

In [8]:
notch_q = 30.0
b_iir, a_iir = iirnotch(hum_frequency_hz, Q=notch_q, fs=fs)
y_iir = lfilter(b_iir, a_iir, x_hum).astype(np.float32)

## 6. Respons frekuensi kedua filter

In [9]:
f_resp, H_fir = freqz(b_fir, a_fir, worN=8192, fs=fs)
_, H_iir = freqz(b_iir, a_iir, worN=8192, fs=fs)
mag_fir_db = 20*np.log10(np.abs(H_fir) + 1e-12)
mag_iir_db = 20*np.log10(np.abs(H_iir) + 1e-12)

## 7. PSD input dan output

In [10]:
nperseg = min(2048, len(x_hum))
f_psd, P_in = welch(x_hum, fs=fs, nperseg=nperseg)
_, P_fir = welch(y_fir, fs=fs, nperseg=nperseg)
_, P_iir = welch(y_iir, fs=fs, nperseg=nperseg)

# Daya lokal di sekitar 50 Hz
def band_power(freq, psd, lo=45.0, hi=55.0):
    mask = (freq >= lo) & (freq <= hi)
    if np.count_nonzero(mask) < 1:
        return float("nan")
    integrate = getattr(np, "trapezoid", np.trapz)
    return float(integrate(psd[mask], freq[mask]))

p50_in = band_power(f_psd, P_in)
p50_fir = band_power(f_psd, P_fir)
p50_iir = band_power(f_psd, P_iir)

def attenuation_db(p_after, p_before):
    return float(10*np.log10((p_after + 1e-20)/(p_before + 1e-20)))

fir_50_db = attenuation_db(p50_fir, p50_in)
iir_50_db = attenuation_db(p50_iir, p50_in)

# Distorsi terhadap audio channel asli pada pita 150 Hz - 4 kHz.
# FIR dikompensasi delay hanya untuk metrik pembanding ini.
Nspec = 8192
Xref = np.abs(np.fft.rfft(x_channel, n=Nspec))
Y_iir = np.abs(np.fft.rfft(y_iir, n=Nspec))
if fir_group_delay_samples < len(y_fir):
    y_fir_aligned = np.pad(
        y_fir[fir_group_delay_samples:],
        (0, fir_group_delay_samples)
    )[:len(x_channel)]
else:
    y_fir_aligned = y_fir
Y_fir = np.abs(np.fft.rfft(y_fir_aligned, n=Nspec))
f_spec = np.fft.rfftfreq(Nspec, 1/fs)
mask_voice = (f_spec >= 150) & (f_spec <= min(4000, fs/2))
ref_db = 20*np.log10(Xref/(Xref.max()+1e-12)+1e-9)
fir_db = 20*np.log10(Y_fir/(Y_fir.max()+1e-12)+1e-9)
iir_db = 20*np.log10(Y_iir/(Y_iir.max()+1e-12)+1e-9)
fir_dist = float(np.sqrt(np.mean((ref_db[mask_voice]-fir_db[mask_voice])**2)))
iir_dist = float(np.sqrt(np.mean((ref_db[mask_voice]-iir_db[mask_voice])**2)))

## 8. Estimasi waktu komputasi relatif di komputer (bukan benchmark MCU)

In [11]:
def bench(b, a, x, repeat=40):
    t0 = time.perf_counter()
    for _ in range(repeat):
        lfilter(b, a, x)
    return (time.perf_counter()-t0)/repeat

fir_time_s = bench(b_fir, a_fir, x_hum)
iir_time_s = bench(b_iir, a_iir, x_hum)

## 9. Folder output

In [12]:
out_audio = ROOT / "data/processed/mp05_filter"
out_feat = ROOT / "features/intermediate"
out_fig = ROOT / "results/figures"
out_met = ROOT / "results/metrics"
for p in [out_audio, out_feat, out_fig, out_met]:
    p.mkdir(parents=True, exist_ok=True)

# Simpan WAV PCM16
def save_pcm16(path, x):
    x = np.asarray(x, dtype=np.float32)
    peak = np.max(np.abs(x)) + 1e-12
    if peak > 0.999:
        x = x * (0.999/peak)
    wavfile.write(path, fs, np.int16(np.clip(x, -1, 1)*32767))

save_pcm16(out_audio / "nyala_001_channel_hum.wav", x_hum)
save_pcm16(out_audio / "nyala_001_fir_hp.wav", y_fir)
save_pcm16(out_audio / "nyala_001_iir_notch.wav", y_iir)

np.savez_compressed(
    out_feat / "mp05_filter_coefficients.npz",
    b_fir=b_fir,
    a_fir=a_fir,
    b_iir=b_iir,
    a_iir=a_iir,
    frequency_hz=f_resp,
    H_fir=H_fir,
    H_iir=H_iir,
)
config = {
    "sample_rate_hz": int(fs),
    "hum_frequency_hz": hum_frequency_hz,
    "hum_amplitude": hum_amplitude,
    "fir": {
        "type": "highpass",
        "numtaps": fir_numtaps,
        "cutoff_hz": fir_cutoff_hz,
        "group_delay_samples": fir_group_delay_samples,
        "group_delay_ms": fir_group_delay_ms,
    },
    "iir": {
        "type": "notch",
        "center_hz": hum_frequency_hz,
        "Q": notch_q,
    },
    "design_stage": "behavior_experiment_not_final_design",
    "recommended_problem_for_mp06": "suppress_50hz_hum_with_minimal_speech_distortion",
}
with open(ROOT / "config/filter_experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
metrics = {
    "power_45_55hz_input": p50_in,
    "power_45_55hz_fir": p50_fir,
    "power_45_55hz_iir": p50_iir,
    "fir_attenuation_50hz_db": fir_50_db,
    "iir_attenuation_50hz_db": iir_50_db,
    "fir_spectral_distortion_150_4000hz_db_rms": fir_dist,
    "iir_spectral_distortion_150_4000hz_db_rms": iir_dist,
    "fir_group_delay_ms": fir_group_delay_ms,
    "fir_estimated_multiplies_per_sample": fir_numtaps,
    "iir_estimated_multiplies_per_sample": 5,
    "fir_pc_processing_time_s": fir_time_s,
    "iir_pc_processing_time_s": iir_time_s,
}
with open(out_met / "mp05_filter_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

## 10. Figure respons filter

In [13]:
fig, ax = plt.subplots(2, 1, figsize=(9, 7))
ax[0].plot(f_resp, mag_fir_db, label="FIR HP 100 Hz")
ax[0].plot(f_resp, mag_iir_db, label="IIR notch 50 Hz")
ax[0].set_xlim(0, min(800, fs/2))
ax[0].set_ylim(-80, 5)
ax[0].set_xlabel("Frequency (Hz)")
ax[0].set_ylabel("Magnitude (dB)")
ax[0].set_title("Magnitude response")
ax[0].legend(); ax[0].grid(alpha=0.2)
ax[1].plot(f_resp, np.unwrap(np.angle(H_fir)), label="FIR")
ax[1].plot(f_resp, np.unwrap(np.angle(H_iir)), label="IIR")
ax[1].set_xlim(0, min(800, fs/2))
ax[1].set_xlabel("Frequency (Hz)")
ax[1].set_ylabel("Phase (rad)")
ax[1].set_title("Phase response")
ax[1].legend(); ax[1].grid(alpha=0.2)
plt.tight_layout()
plt.savefig(out_fig / "mp05_filter_response.png", dpi=180)
plt.close()

## 11. Figure efek pada PSD

In [14]:
fig, ax = plt.subplots(2, 1, figsize=(9, 7))
show = min(len(x_hum), int(0.20*fs))
t = np.arange(show)/fs
ax[0].plot(t, x_hum[:show], label="channel + hum", linewidth=0.8)
ax[0].plot(t, y_iir[:show], label="IIR notch", linewidth=0.8, alpha=0.85)
ax[0].set_title("Waveform comparison")
ax[0].set_xlabel("Time (s)")
ax[0].legend(); ax[0].grid(alpha=0.2)
ax[1].plot(f_psd, 10*np.log10(P_in+1e-18), label="input")
ax[1].plot(f_psd, 10*np.log10(P_fir+1e-18), label="FIR")
ax[1].plot(f_psd, 10*np.log10(P_iir+1e-18), label="IIR")
ax[1].set_xlim(0, min(1000, fs/2))
ax[1].set_xlabel("Frequency (Hz)")
ax[1].set_ylabel("PSD (dB/Hz)")
ax[1].set_title("PSD before and after filtering")
ax[1].legend(); ax[1].grid(alpha=0.2)
plt.tight_layout()
plt.savefig(out_fig / "mp05_filter_effect.png", dpi=180)
plt.close()

## 12. Handoff ke Mini Project 6

In [15]:
status.update({
    "status": "ready_for_mp06",
    "last_completed_mini_project": 5,
    "filter_experiment_config": "config/filter_experiment_config.json",
    "filter_coefficients": "features/intermediate/mp05_filter_coefficients.npz",
    "filter_metrics": "results/metrics/mp05_filter_metrics.json",
    "filter_input_hum": "data/processed/mp05_filter/nyala_001_channel_hum.wav",
    "filter_output_fir": "data/processed/mp05_filter/nyala_001_fir_hp.wav",
    "filter_output_iir": "data/processed/mp05_filter/nyala_001_iir_notch.wav",
    "mp06_design_problem": "suppress_50hz_hum_with_minimal_speech_distortion",
})
with open(STATUS_PATH, "w", encoding="utf-8") as f:
    json.dump(status, f, indent=2)

print("Mini Project 5 selesai")
print("FIR attenuation 45-55 Hz (dB):", round(fir_50_db, 2))
print("IIR attenuation 45-55 Hz (dB):", round(iir_50_db, 2))
print("FIR spectral distortion RMS (dB):", round(fir_dist, 2))
print("IIR spectral distortion RMS (dB):", round(iir_dist, 2))
print("Status: ready_for_mp06")

Mini Project 5 selesai
FIR attenuation 45-55 Hz (dB): -9.55
IIR attenuation 45-55 Hz (dB): -15.01
FIR spectral distortion RMS (dB): 36.54
IIR spectral distortion RMS (dB): 39.27
Status: ready_for_mp06


#### FITUR TAMBAHAN - FILTER STABILITY, DELAY & EMBEDDED COST
##### Deskripsi Fitur Tambahan (Sesuai Karakteristik Mini Project 5):
##### 1. Analisis Pole-Zero IIR Notch: radius pole, radius zero, margin stabilitas.
##### 2. Bandwidth -3 dB Notch: hasil ukur (resolusi 0.01 Hz) vs teori BW = f0 / Q.
##### 3. Analisis Transition Band FIR: gain FIR dan IIR pada 50, 100, 150, 200, 300 Hz.
##### 4. Group Delay: FIR linear phase (konstan) vs IIR (melonjak di sekitar 50 Hz).
##### 5. Transien IIR: settling time respons impuls hingga turun 60 dB (ukur vs teori).
##### 6. SNR terhadap audio channel bersih: input ber-hum, output FIR, output IIR.
##### 7. Perubahan energi pita rendah ucapan 80-300 Hz akibat filtering.
##### 8. Estimasi biaya embedded (float32): memori koefisien+state dan MAC per detik.


In [19]:
print("\n" + "=" * 58)
print("FITUR TAMBAHAN - FILTER STABILITY, DELAY & EMBEDDED COST")
print("=" * 58)

# 1. ANALISIS POLE-ZERO & STABILITAS IIR NOTCH
z_iir, p_iir, k_iir = tf2zpk(b_iir, a_iir)
pole_radius = float(np.max(np.abs(p_iir)))
zero_radius = float(np.max(np.abs(z_iir)))
stability_margin = float(1.0 - pole_radius)
iir_is_stable = bool(pole_radius < 1.0)

# 2. BANDWIDTH -3 dB NOTCH (RESOLUSI HALUS 0.01 Hz)
f_fine = np.linspace(30.0, 70.0, 4001)
_, H_iir_fine = freqz(b_iir, a_iir, worN=f_fine, fs=fs)
mag_iir_fine_db = 20*np.log10(np.abs(H_iir_fine) + 1e-12)
below_3db = f_fine[mag_iir_fine_db <= -3.0]
if len(below_3db) > 0:
    notch_bw_measured_hz = float(below_3db.max() - below_3db.min())
else:
    notch_bw_measured_hz = float("nan")
notch_bw_theory_hz = float(hum_frequency_hz / notch_q)
notch_depth_db = float(np.min(mag_iir_fine_db))

# 3. ANALISIS TRANSITION BAND FIR VS IIR
f_check = np.array([50.0, 100.0, 150.0, 200.0, 300.0])
_, H_fir_check = freqz(b_fir, a_fir, worN=f_check, fs=fs)
_, H_iir_check = freqz(b_iir, a_iir, worN=f_check, fs=fs)
fir_gain_at_db = {f"{int(fc)}_hz": float(20*np.log10(np.abs(h) + 1e-12))
                  for fc, h in zip(f_check, H_fir_check)}
iir_gain_at_db = {f"{int(fc)}_hz": float(20*np.log10(np.abs(h) + 1e-12))
                  for fc, h in zip(f_check, H_iir_check)}

# 4. GROUP DELAY FIR VS IIR
f_gd = np.linspace(1.0, fs/2, 16000)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _, gd_fir = group_delay((b_fir, a_fir), w=f_gd, fs=fs)
    _, gd_iir = group_delay((b_iir, a_iir), w=f_gd, fs=fs)
gd_fir_ms = np.where(np.isfinite(gd_fir), gd_fir, np.nan) * 1000.0 / fs
gd_iir_ms = np.where(np.isfinite(gd_iir), gd_iir, np.nan) * 1000.0 / fs
mask_gd_voice = (f_gd >= 150) & (f_gd <= min(4000, fs/2))
mask_gd_low = (f_gd >= 20) & (f_gd <= 200)
fir_gd_voice_mean_ms = float(np.nanmean(gd_fir_ms[mask_gd_voice]))
fir_gd_voice_std_ms = float(np.nanstd(gd_fir_ms[mask_gd_voice]))
iir_gd_peak_ms = float(np.nanmax(np.abs(gd_iir_ms[mask_gd_low])))
iir_gd_voice_mean_ms = float(np.nanmean(np.abs(gd_iir_ms[mask_gd_voice])))

# 5. TRANSIEN / SETTLING TIME IIR (RESPONS IMPULS TURUN 60 dB)
imp_len = int(3.0 * fs)
delta = np.zeros(imp_len)
delta[0] = 1.0
h_iir = lfilter(b_iir, a_iir, delta)
tail = np.abs(h_iir[1:])
tail_peak = float(np.max(tail)) + 1e-20
above = np.where(tail >= 1e-3 * tail_peak)[0]
settling_ms_measured = float((above[-1] + 1) * 1000.0 / fs) if len(above) else 0.0
settling_ms_theory = float(1000.0 * np.log(1e-3) / np.log(pole_radius) / fs)

# 6. SNR TERHADAP AUDIO CHANNEL BERSIH
peak_raw = float(np.max(np.abs(x_channel + hum)) + 1e-12)
gain_used = 0.98 / peak_raw if peak_raw > 0.98 else 1.0
x_ref = (x_channel * gain_used).astype(np.float64)

def snr_vs_ref(ref, test, start=0):
    n_len = min(len(ref), len(test))
    r = ref[start:n_len]
    e = r - np.asarray(test[start:n_len], dtype=np.float64)
    return float(10*np.log10((np.mean(r**2) + 1e-12) / (np.mean(e**2) + 1e-12)))

skip = min(int(settling_ms_theory * fs / 1000.0), len(x_ref) // 2)
snr_input_db = snr_vs_ref(x_ref, x_hum)
snr_fir_db = snr_vs_ref(x_ref, y_fir_aligned)
snr_iir_db = snr_vs_ref(x_ref, y_iir)
snr_iir_after_settling_db = snr_vs_ref(x_ref, y_iir, start=skip)

# 7. PERUBAHAN ENERGI PITA RENDAH UCAPAN (80-300 Hz)
f_w, P_ref = welch(x_ref, fs=fs, nperseg=nperseg)
_, P_fir_al = welch(y_fir_aligned, fs=fs, nperseg=nperseg)
low_ref = band_power(f_w, P_ref, 80.0, 300.0)
low_fir = band_power(f_w, P_fir_al, 80.0, 300.0)
low_iir = band_power(f_psd, P_iir, 80.0, 300.0)
fir_low_band_change_db = attenuation_db(low_fir, low_ref)
iir_low_band_change_db = attenuation_db(low_iir, low_ref)

# 8. ESTIMASI BIAYA EMBEDDED (FLOAT32)
bytes_per_value = 4
fir_mac_per_sample = fir_numtaps
iir_mac_per_sample = 5
fir_memory_bytes = (fir_numtaps + (fir_numtaps - 1)) * bytes_per_value  # koef + state
iir_memory_bytes = (5 + 2) * bytes_per_value  # 5 koef biquad + 2 state (DF2T)
fir_mmac_per_s = fir_mac_per_sample * fs / 1e6
iir_mmac_per_s = iir_mac_per_sample * fs / 1e6

# 9. SIMPAN METRIK FITUR TAMBAHAN KE JSON
ext_metrics = {
    "feature_extension": "Filter Stability, Delay & Embedded Cost",
    "iir_pole_zero": {
        "pole_radius": pole_radius,
        "zero_radius": zero_radius,
        "stability_margin": stability_margin,
        "is_stable": iir_is_stable
    },
    "iir_notch_bandwidth": {
        "bw_3db_measured_hz": notch_bw_measured_hz,
        "bw_3db_theory_hz": notch_bw_theory_hz,
        "notch_depth_db": notch_depth_db
    },
    "transition_band_gain_db": {
        "fir": fir_gain_at_db,
        "iir": iir_gain_at_db
    },
    "group_delay": {
        "fir_voice_mean_ms": fir_gd_voice_mean_ms,
        "fir_voice_std_ms": fir_gd_voice_std_ms,
        "iir_peak_20_200hz_ms": iir_gd_peak_ms,
        "iir_voice_mean_ms": iir_gd_voice_mean_ms
    },
    "iir_transient": {
        "settling_60db_measured_ms": settling_ms_measured,
        "settling_60db_theory_ms": settling_ms_theory
    },
    "snr_vs_clean_channel_db": {
        "input_with_hum": snr_input_db,
        "fir_output": snr_fir_db,
        "iir_output": snr_iir_db,
        "iir_output_after_settling": snr_iir_after_settling_db
    },
    "speech_low_band_80_300hz_change_db": {
        "fir": fir_low_band_change_db,
        "iir": iir_low_band_change_db
    },
    "embedded_cost_float32": {
        "fir_mac_per_sample": fir_mac_per_sample,
        "iir_mac_per_sample": iir_mac_per_sample,
        "fir_memory_bytes": fir_memory_bytes,
        "iir_memory_bytes": iir_memory_bytes,
        "fir_mmac_per_second": fir_mmac_per_s,
        "iir_mmac_per_second": iir_mmac_per_s
    }
}
ext_met_path = out_met / "mp05_extension_filter_metrics.json"
with open(ext_met_path, "w", encoding="utf-8") as f:
    json.dump(ext_metrics, f, indent=2)

# 10. VISUALISASI FITUR TAMBAHAN (4 SUBPLOT)
fig_ext, ax_ext = plt.subplots(2, 2, figsize=(11, 8.5))

# Subplot 1: Pole-zero IIR (zoom dekat z = 1)
theta = np.linspace(0, 2*np.pi, 4000)
ax_ext[0, 0].plot(np.cos(theta), np.sin(theta), color="gray", linewidth=0.8, label="Lingkaran satuan")
ax_ext[0, 0].plot(z_iir.real, z_iir.imag, "o", markerfacecolor="none", markeredgecolor="navy", markersize=9, label="Zero")
ax_ext[0, 0].plot(p_iir.real, p_iir.imag, "x", color="crimson", markersize=9, label="Pole")
ax_ext[0, 0].set_xlim(0.97, 1.01)
ax_ext[0, 0].set_ylim(-0.03, 0.03)
ax_ext[0, 0].set_aspect("equal")
ax_ext[0, 0].set_title(f"Extension 1: Pole-Zero IIR (zoom) | r_pole = {pole_radius:.5f}")
ax_ext[0, 0].set_xlabel("Real")
ax_ext[0, 0].set_ylabel("Imag")
ax_ext[0, 0].legend(fontsize=8, loc="upper left")
ax_ext[0, 0].grid(alpha=0.25)

# Subplot 2: Magnitude zoom 0-400 Hz (transition band FIR vs notch IIR)
ax_ext[0, 1].plot(f_resp, mag_fir_db, label="FIR HP 100 Hz", linewidth=1.0)
ax_ext[0, 1].plot(f_resp, mag_iir_db, label="IIR notch 50 Hz", linewidth=1.0)
ax_ext[0, 1].axvline(50, color="crimson", linestyle="--", linewidth=0.8, label="Hum 50 Hz")
ax_ext[0, 1].axvline(100, color="darkorange", linestyle=":", linewidth=0.9, label="Cutoff FIR 100 Hz")
ax_ext[0, 1].set_xlim(0, 400)
ax_ext[0, 1].set_ylim(-60, 5)
ax_ext[0, 1].set_title("Extension 2: Transition Band FIR vs Notch IIR (0-400 Hz)")
ax_ext[0, 1].set_xlabel("Frekuensi (Hz)")
ax_ext[0, 1].set_ylabel("Magnitudo (dB)")
ax_ext[0, 1].legend(fontsize=8, loc="lower right")
ax_ext[0, 1].grid(alpha=0.25)

# Subplot 3: Group delay
ax_ext[1, 0].plot(f_gd, gd_fir_ms, label=f"FIR (~{fir_gd_voice_mean_ms:.2f} ms konstan)", linewidth=1.0)
ax_ext[1, 0].plot(f_gd, gd_iir_ms, label="IIR notch", linewidth=1.0)
ax_ext[1, 0].set_xlim(0, 400)
ax_ext[1, 0].set_title(f"Extension 3: Group Delay | Puncak IIR = {iir_gd_peak_ms:.1f} ms")
ax_ext[1, 0].set_xlabel("Frekuensi (Hz)")
ax_ext[1, 0].set_ylabel("Group delay (ms)")
ax_ext[1, 0].legend(fontsize=8, loc="upper right")
ax_ext[1, 0].grid(alpha=0.25)

# Subplot 4: Peluruhan respons impuls IIR
t_imp_ms = np.arange(1, imp_len) * 1000.0 / fs
h_db = 20*np.log10(tail / tail_peak + 1e-12)
ax_ext[1, 1].plot(t_imp_ms, h_db, color="purple", linewidth=0.6, label="|h[n]| IIR (relatif)")
ax_ext[1, 1].axhline(-60, color="crimson", linestyle="--", linewidth=0.9, label="-60 dB")
ax_ext[1, 1].axvline(settling_ms_measured, color="darkorange", linestyle=":", linewidth=1.0,
                     label=f"Settling = {settling_ms_measured:.0f} ms")
ax_ext[1, 1].set_ylim(-80, 3)
ax_ext[1, 1].set_title("Extension 4: Transien Respons Impuls IIR Notch")
ax_ext[1, 1].set_xlabel("Waktu (ms)")
ax_ext[1, 1].set_ylabel("Magnitudo (dB)")
ax_ext[1, 1].legend(fontsize=8, loc="upper right")
ax_ext[1, 1].grid(alpha=0.25)

fig_ext.tight_layout()
ext_fig_path = out_fig / "mp05_extension_filter_analysis.png"
fig_ext.savefig(ext_fig_path, dpi=180)
plt.close(fig_ext)

# 11. OUTPUT TERMINAL FITUR TAMBAHAN
print("Pole radius IIR              :", round(pole_radius, 6), "| stabil:", iir_is_stable)
print("Bandwidth -3 dB notch (ukur) :", round(notch_bw_measured_hz, 3), "Hz")
print("Bandwidth -3 dB notch (teori):", round(notch_bw_theory_hz, 3), "Hz")
print("Gain FIR di 50 Hz            :", round(fir_gain_at_db["50_hz"], 2), "dB")
print("Gain FIR di 100 Hz           :", round(fir_gain_at_db["100_hz"], 2), "dB")
print("Group delay FIR (pita suara) :", round(fir_gd_voice_mean_ms, 3), "ms")
print("Puncak group delay IIR       :", round(iir_gd_peak_ms, 2), "ms")
print("Settling time IIR (ukur)     :", round(settling_ms_measured, 1), "ms")
print("SNR input ber-hum            :", round(snr_input_db, 2), "dB")
print("SNR output FIR               :", round(snr_fir_db, 2), "dB")
print("SNR output IIR               :", round(snr_iir_db, 2), "dB")
print("SNR IIR setelah settling     :", round(snr_iir_after_settling_db, 2), "dB")
print("Perubahan 80-300 Hz FIR / IIR:", round(fir_low_band_change_db, 2), "/",
      round(iir_low_band_change_db, 2), "dB")
print("Memori FIR / IIR (float32)   :", fir_memory_bytes, "/", iir_memory_bytes, "bytes")
print("Beban FIR / IIR              :", round(fir_mmac_per_s, 3), "/", round(iir_mmac_per_s, 3), "MMAC/s")
print("Extension Metrics File       :", ext_met_path)
print("Extension Figure File        :", ext_fig_path)
print("=" * 58)



FITUR TAMBAHAN - FILTER STABILITY, DELAY & EMBEDDED COST
Pole radius IIR              : 0.999673 | stabil: True
Bandwidth -3 dB notch (ukur) : 1.66 Hz
Bandwidth -3 dB notch (teori): 1.667 Hz
Gain FIR di 50 Hz            : -9.55 dB
Gain FIR di 100 Hz           : -5.94 dB
Group delay FIR (pita suara) : 4.0 ms
Puncak group delay IIR       : 177.91 ms
Settling time IIR (ukur)     : 1311.1 ms
SNR input ber-hum            : 13.27 dB
SNR output FIR               : 12.98 dB
SNR output IIR               : 26.99 dB
SNR IIR setelah settling     : 40.81 dB
Perubahan 80-300 Hz FIR / IIR: -2.67 / -0.0 dB
Memori FIR / IIR (float32)   : 1028 / 28 bytes
Beban FIR / IIR              : 2.064 / 0.08 MMAC/s
Extension Metrics File       : C:\Users\Erick W\OneDrive\Documents\Python\Mini Project 05\results\metrics\mp05_extension_filter_metrics.json
Extension Figure File        : C:\Users\Erick W\OneDrive\Documents\Python\Mini Project 05\results\figures\mp05_extension_filter_analysis.png
